<a href="https://colab.research.google.com/github/shreyanshsamaje7-prog/BIS_LAB/blob/main/Grey_wolf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np

# Hospital departments
departments = ["Emergency", "Cardiology", "Neurology"]

# Total available resources
total = np.array([30, 60, 45])

# Patient demand (illustrative)
demand = np.array([120, 100, 80])

# GWO parameters
wolves = 20
iterations = 100
D = 9

# Fitness function
def fitness(x):
    x = np.maximum(1, x.reshape(3, 3))

    doctors = x[:, 0]
    nurses = x[:, 1]
    equipment = x[:, 2]

    capacity = doctors * 5 + nurses * 2 + equipment * 1

    waiting = np.mean(demand / capacity)

    utilization = np.mean(
        np.minimum(demand, capacity) / capacity
    )

    workload = np.var(demand / (doctors + nurses))

    return waiting + 0.5 * (1 - utilization) + 0.001 * workload


# Repair resource constraints
def repair(x):
    x = np.maximum(1, np.round(x.reshape(3, 3)))

    for j in range(3):
        while x[:, j].sum() > total[j]:
            i = np.argmax(x[:, j])
            if x[i, j] > 1:
                x[i, j] -= 1
            else:
                break

    return x.flatten()


# Initialize wolves
X = np.random.uniform(2, 15, (wolves, D))
X = np.array([repair(x) for x in X])

# Grey Wolf Optimization
for t in range(iterations):

    scores = np.array([fitness(x) for x in X])
    order = np.argsort(scores)

    alpha = X[order[0]].copy()
    beta = X[order[1]].copy()
    delta = X[order[2]].copy()

    a = 2 - 2 * t / iterations

    for i in range(wolves):
        new_positions = []

        for leader in [alpha, beta, delta]:
            A = 2 * a * np.random.rand(D) - a
            C = 2 * np.random.rand(D)

            distance = abs(C * leader - X[i])
            new_positions.append(leader - A * distance)

        X[i] = repair(np.mean(new_positions, axis=0))

# Best solution
scores = np.array([fitness(x) for x in X])
best = X[np.argmin(scores)].reshape(3, 3)

print("Optimal Hospital Resource Allocation")
for i, name in enumerate(departments):
    print(name, "Doctors:", int(best[i, 0]),
          "Nurses:", int(best[i, 1]),
          "Equipment:", int(best[i, 2]))

print("Best Fitness:", min(scores))


Optimal Hospital Resource Allocation
Emergency Doctors: 11 Nurses: 25 Equipment: 15
Cardiology Doctors: 10 Nurses: 21 Equipment: 15
Neurology Doctors: 9 Nurses: 14 Equipment: 15
Best Fitness: 0.9739557579743403
